# Ghost of the Machine — baseline

A trivial reference baseline: estimate one number from `dataset/train/` — the average
position of the boundary as a fraction of the passage length — and predict that
same fraction of the length for every test passage.

It exists as a runnable template for the contract: read `dataset/test_public/data.jsonl`,
write `answers.jsonl` at the repository root. Replace the logic below with
your own method.


In [ ]:
import json, os

TRAIN_DIR = "dataset/train"
TEST_DIR  = "dataset/test_public"
OUTPUT    = "answers.jsonl"

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

# ---- "train": average boundary position as a fraction of passage length ----
train_rows = read_jsonl(f"{TRAIN_DIR}/data.jsonl")
train_ans = {r["id"]: r["boundary_char_index"] for r in read_jsonl(f"{TRAIN_DIR}/answers.jsonl")}
fracs = [train_ans[r["id"]] / len(r["text"]) for r in train_rows if len(r["text"]) > 0]
mean_frac = sum(fracs) / len(fracs)
print(f"mean boundary fraction from {len(fracs)} train passages: {mean_frac:.4f}")

# ---- predict: same fraction for every test passage ----
test_rows = read_jsonl(f"{TEST_DIR}/data.jsonl")
preds = {r["id"]: int(mean_frac * len(r["text"])) for r in test_rows}

with open(OUTPUT, "w", encoding="utf-8") as f:
    for r in test_rows:
        f.write(json.dumps({"id": r["id"], "boundary_char_index": preds[r["id"]]}) + "\n")
print(f"wrote {OUTPUT}: {len(preds)} predictions")

# self-score when the dev answers are present (absent in the hidden grading set)
ans_path = f"{TEST_DIR}/answers.jsonl"
if os.path.exists(ans_path):
    import math
    gt = {r["id"]: r["boundary_char_index"] for r in read_jsonl(ans_path)}
    scores = [math.exp(-abs(preds[i] - gt[i]) / 100.0) for i in gt if i in preds]
    print(f"self-score: {sum(scores)/len(scores):.4f}")